In [ ]:
# Phase 4 — Multi-Agent Orchestration

This notebook builds OurBrain's multi-agent system on top of the MCP server from Phase 2.

**Architecture:**
- `agents/mcp_host.py` — custom MCP host that connects to ourbrain_server.py
- `agents/agent_runner.py` — agentic loop with tool scoping
- `agents/agents.py` — three specialized sub-agents (Nutrition, Planner, Suggestion)
- `agents/orchestrator.py` — routes questions and synthesizes responses

**What this demonstrates:** custom MCP host implementation, the agentic loop pattern, 
tool scoping per agent, and orchestrator-worker multi-agent design.

In [3]:
import sys

# Add the project root (one level up from the notebook) to Python's import path
# This lets Python find the `agents` package
if ".." not in sys.path:
    sys.path.insert(0, "..")

# If anything from the old in-notebook version is cached, clear it
for mod in list(sys.modules.keys()):
    if mod.startswith("agents"):
        del sys.modules[mod]

from agents.mcp_host import OurBrainMCPHost
print("Imported OurBrainMCPHost from agents.mcp_host ✓")

Imported OurBrainMCPHost from agents.mcp_host ✓


In [4]:
host = OurBrainMCPHost(server_path="../mcp_server/ourbrain_server.py")

tools = await host.list_tools()
for t in tools:
    print(f"- {t.name}: {t.description}")

- get_meals: Get all meals stored in OurKitchen, including name, cuisine, proteins, and ingredients.
- get_meal_history: Get the full weekly meal history, with each planned meal enriched with its recipe details.
- get_preferences: Get household dietary preferences and restrictions.
- get_groceries: Get the current grocery list.


In [ ]:
## Piece 2 — Agent Runner

Built `agents/agent_runner.py` containing `run_agent()` — the core agentic loop.

This function takes a Claude conversation and a set of MCP tools, lets Claude 
decide which tools to call, executes them via our MCP host, feeds results back, 
and loops until Claude has a final answer. Includes a `max_turns` safety limit 
to prevent runaway loops.

**Concepts demonstrated:** the agentic loop, tool scoping per agent, 
async tool execution.

In [5]:
agent_runner_code = '''"""
Agent runner for OurBrain.

The core agentic loop: send Claude a message with scoped MCP tools,
let Claude decide which to call, execute them via the MCP host, feed
results back, and repeat until Claude has a final answer.

This is the pattern that powers every agent system: Cursor, Claude Code,
LangChain agents, OpenAI Assistants. The implementation differs but the
loop is the same.
"""

import json
from anthropic import Anthropic
from agents.mcp_host import OurBrainMCPHost


def _mcp_tools_to_anthropic_format(mcp_tools, allowed_names=None):
    """
    Convert MCP tool definitions to Anthropic API format.
    
    If allowed_names is provided, filter to only those tools.
    This is the 'tool scoping per agent' pattern that makes
    multi-agent specialization work.
    """
    converted = []
    for t in mcp_tools:
        if allowed_names is not None and t.name not in allowed_names:
            continue
        converted.append({
            "name": t.name,
            "description": t.description,
            "input_schema": t.inputSchema,
        })
    return converted


async def run_agent(
    host: OurBrainMCPHost,
    anthropic_client: Anthropic,
    model: str,
    system_prompt: str,
    user_question: str,
    allowed_tools: list[str] | None = None,
    max_turns: int = 6,
    verbose: bool = True,
) -> str:
    """
    Run a single sub-agent: a Claude conversation with scoped MCP tools.
    
    Loops until Claude returns a final text answer, or until max_turns
    is hit (a safety net against runaway tool-calling loops).
    
    Returns Claude's final text response as a string.
    """
    all_mcp_tools = await host.list_tools()
    tools = _mcp_tools_to_anthropic_format(all_mcp_tools, allowed_names=allowed_tools)
    
    if verbose:
        scoped = [t["name"] for t in tools]
        print(f"  [agent has access to: {scoped}]")
    
    messages = [{"role": "user", "content": user_question}]
    
    for turn in range(max_turns):
        response = anthropic_client.messages.create(
            model=model,
            max_tokens=2048,
            system=system_prompt,
            tools=tools,
            messages=messages,
        )
        
        if response.stop_reason == "end_turn":
            text_blocks = [b.text for b in response.content if b.type == "text"]
            return "\\n".join(text_blocks)
        
        if response.stop_reason == "tool_use":
            messages.append({"role": "assistant", "content": response.content})
            
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    if verbose:
                        print(f"  [tool call: {block.name}({block.input})]")
                    result = await host.call_tool(block.name, block.input)
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": result,
                    })
            
            messages.append({"role": "user", "content": tool_results})
            continue
        
        return f"[unexpected stop_reason: {response.stop_reason}]"
    
    return "[hit max_turns without a final answer]"
'''

with open("../agents/agent_runner.py", "w") as f:
    f.write(agent_runner_code)

print("Wrote ../agents/agent_runner.py")

Wrote ../agents/agent_runner.py


In [6]:
# Force fresh imports in case anything is cached
import sys
for mod in list(sys.modules.keys()):
    if mod.startswith("agents"):
        del sys.modules[mod]

from agents.agent_runner import run_agent
from anthropic import Anthropic
import json

# Load API key (in case this cell is run on its own)
with open("/Users/sagar/dev/TheBrain/secrets.json") as f:
    secrets = json.load(f)

anthropic_client = Anthropic(api_key=secrets["ANTHROPIC_API_KEY"])
MODEL = "claude-sonnet-4-5"  # if model-not-found, change to "claude-sonnet-4-20250514"

print("Setup complete ✓")

Setup complete ✓


In [7]:
answer = await run_agent(
    host=host,
    anthropic_client=anthropic_client,
    model=MODEL,
    system_prompt="You are a helpful assistant for the OurKitchen household app. Be concise.",
    user_question="What dietary restrictions does this household have?",
)

print("\n--- ANSWER ---")
print(answer)

  [agent has access to: ['get_meals', 'get_meal_history', 'get_preferences', 'get_groceries']]


BadRequestError: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'messages.2: user messages must have non-empty content'}, 'request_id': 'req_011CaYkUKZrK4VYn5WfaV5iN'}

In [8]:
# Test each tool directly to see what they return
for tool_name in ["get_meals", "get_meal_history", "get_preferences", "get_groceries"]:
    result = await host.call_tool(tool_name)
    print(f"{tool_name}: type={type(result).__name__}, len={len(result) if result else 0}")
    print(f"  preview: {result[:100] if result else '(empty)'}")
    print()

get_meals: type=str, len=10346
  preview: [{"cuisine": "Vietnamese", "note": "", "fav": false, "rating": 5, "ts": 1774991541622, "proteins": [

get_meal_history: type=str, len=6452
  preview: [{"week_of": "2026-03-30", "meals": [{"date": "2026-03-31", "slot": "Dinner", "meal_name": "Unknown"

get_preferences: type=str, len=52
  preview: {"main": {"restrictions": ["no pork"], "loves": []}}

get_groceries: type=str, len=4121
  preview: [{"qty": "", "id": "mnq4qr1938f3un95ydq", "cat": "Other", "name": "Sunflower seeds", "checked": fals



In [9]:
agent_runner_code = '''"""
Agent runner for OurBrain.

The core agentic loop: send Claude a message with scoped MCP tools,
let Claude decide which to call, execute them via the MCP host, feed
results back, and repeat until Claude has a final answer.
"""

from anthropic import Anthropic
from agents.mcp_host import OurBrainMCPHost


def _mcp_tools_to_anthropic_format(mcp_tools, allowed_names=None):
    """Convert MCP tool definitions to Anthropic API format, optionally filtered."""
    converted = []
    for t in mcp_tools:
        if allowed_names is not None and t.name not in allowed_names:
            continue
        converted.append({
            "name": t.name,
            "description": t.description,
            "input_schema": t.inputSchema,
        })
    return converted


async def run_agent(
    host: OurBrainMCPHost,
    anthropic_client: Anthropic,
    model: str,
    system_prompt: str,
    user_question: str,
    allowed_tools: list[str] | None = None,
    max_turns: int = 6,
    verbose: bool = True,
) -> str:
    """
    Run a single sub-agent: a Claude conversation with scoped MCP tools.
    """
    all_mcp_tools = await host.list_tools()
    tools = _mcp_tools_to_anthropic_format(all_mcp_tools, allowed_names=allowed_tools)
    
    if verbose:
        scoped = [t["name"] for t in tools]
        print(f"  [agent has access to: {scoped}]")
    
    messages = [{"role": "user", "content": user_question}]
    
    for turn in range(max_turns):
        response = anthropic_client.messages.create(
            model=model,
            max_tokens=2048,
            system=system_prompt,
            tools=tools,
            messages=messages,
        )
        
        if verbose:
            print(f"  [turn {turn + 1}: stop_reason={response.stop_reason}]")
        
        # Case 1: Claude is done
        if response.stop_reason == "end_turn":
            text_blocks = [b.text for b in response.content if b.type == "text"]
            return "\\n".join(text_blocks)
        
        # Case 2: Claude wants to call tools
        if response.stop_reason == "tool_use":
            messages.append({"role": "assistant", "content": response.content})
            
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    if verbose:
                        print(f"  [tool call: {block.name}({block.input})]")
                    result = await host.call_tool(block.name, block.input)
                    
                    # Defensive: replace empty results with a sentinel so
                    # Claude knows the tool returned nothing rather than failing
                    if not result or not result.strip():
                        result = "(no data returned)"
                    
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": result,
                    })
            
            # Defensive: if no tool calls were actually present despite the
            # stop_reason, bail rather than sending an empty user message
            if not tool_results:
                return "[stop_reason was tool_use but no tool calls were found]"
            
            messages.append({"role": "user", "content": tool_results})
            continue
        
        # Any other stop reason
        return f"[unexpected stop_reason: {response.stop_reason}]"
    
    return "[hit max_turns without a final answer]"
'''

with open("../agents/agent_runner.py", "w") as f:
    f.write(agent_runner_code)

print("Wrote updated ../agents/agent_runner.py")

Wrote updated ../agents/agent_runner.py


In [10]:
import sys
for mod in list(sys.modules.keys()):
    if mod.startswith("agents"):
        del sys.modules[mod]

from agents.agent_runner import run_agent

answer = await run_agent(
    host=host,
    anthropic_client=anthropic_client,
    model=MODEL,
    system_prompt="You are a helpful assistant for the OurKitchen household app. Be concise.",
    user_question="What dietary restrictions does this household have?",
)

print("\n--- ANSWER ---")
print(answer)

  [agent has access to: ['get_meals', 'get_meal_history', 'get_preferences', 'get_groceries']]
  [turn 1: stop_reason=tool_use]
  [tool call: get_preferences({})]
  [turn 2: stop_reason=end_turn]

--- ANSWER ---
This household has one dietary restriction: **no pork**.


In [11]:
# Test 2 — forces multiple tool calls
print("=" * 60)
print("TEST: multi-tool reasoning")
print("=" * 60)
answer = await run_agent(
    host=host,
    anthropic_client=anthropic_client,
    model=MODEL,
    system_prompt="You are a helpful assistant for the OurKitchen household app. Be concise.",
    user_question="What have we cooked the most over the last month, and does it align with our preferences?",
)
print(f"\nANSWER: {answer}\n")

TEST: multi-tool reasoning
  [agent has access to: ['get_meals', 'get_meal_history', 'get_preferences', 'get_groceries']]
  [turn 1: stop_reason=tool_use]
  [tool call: get_meal_history({})]
  [tool call: get_preferences({})]
  [turn 2: stop_reason=end_turn]

ANSWER: Based on your meal history, here's what you've cooked the most over the last month:

**Most Frequent Meals:**
1. **Chicken Alfredo with Broccoli** - 2 times (Italian)
2. **Ground Beef Shawarma** - 2 times (Greek)
3. **Bibimbap** - 2 times (Asian)
4. **Chicken Quesadillas** - 2 times (Mexican)
5. **Pesto Chicken Pasta** - 2 times (Italian)

**Cuisine Breakdown:**
- Italian: 4 meals
- Asian/Greek/Mexican: 2 meals each

**Protein Breakdown:**
- Chicken: 6 meals
- Beef: 5 meals
- Seafood (Fish/Shrimp): 1 meal

**Alignment with Preferences:**
✅ **Yes, this aligns perfectly!** Your household restriction is "no pork," and none of the meals you've cooked contain pork. You've focused primarily on chicken and beef, which fits well w